## Model of "Colonnade: A Reconfigurable SRAM-Based Digital Bit-Serial Compute-In-Memory Macro for Processing Neural Networks", JSSC 2021

Paper by Hyunjoon Kim, Taegeun Yoo, Tony Tae-Hyoung Kim, and Bongjin Kim

## Description of The Macro

Colonnade is an SRAM-based digital CiM macro that supports variable-precision
(1-16b) inputs and weights. Its fully-digital implementation allows it to
compute MAC operations with full precision (*i.e.,* no quantization error and no
loss of fidelity). The macro uses a 128-row by 128-column SRAM array.

Note that our model and language here are different than that of the paper. In
the paper, outputs are reused between columns and inputs are reused between
rows. For consistency with the other models, we have chosen to transpose the
array such that outputs are reused between rows and inputs are reused between
columns. **In the language below, we will refer to the array as if it were
transposed in order to match our models. For a true representation of the
paper's fabricated chip, swap rows for columns and vice versa.**

To support in-array digital computation, every SRAM bitcell is augmented with a
MUX (to reconfigure), XNOR gate (to perform a 1b multiply), and a full adder (to
accumulate). The macro is pipelined to increase throughput; every eight array
rows are separated by registers (flip flops) such that the critical path length
only traverses the full adders of eight rows.

To support variable-precision inputs, each input bit is processed in a separate
cycle. To support variable-precision weights, weights may be stored in 1-16
columns, one column for each weight bit. An additional log2(#rows) bits are
needed for each weight (7 with the 128-row array) because the result of #rows
MAC operations are accumulated in the array, and accumulated outputs may be up
to 7 bits larger by the time they reach the last row of the array.

### Macro Level

- *Input Path*: Row drivers supply one input bit at a time to the array. Inputs
  are provided to the input ports of the array's digital logic. We charge for
  the per-cell digital logic here because one input will be sent to ALL cells in
  a row (because they are wired together) regardless of whether the cell is used
  for a computation or not.
- *Weight Path*: Weight drivers are used to rewrite weights in the array.
- *Output Path*: Column drivers activate columns for reading.

Next, there are 128 columns. Each column stores either a vector of weight bits
or is set to zero to allow for MAC results to spill over into that column.
Inputs are reused between columns (*i.e.,* each input-carrying wire connects to
all columns), while outputs and weights are not reused.

### Column Level

- *Input Path*: Each input is passed directly to a row group in the column.
- *Weight Path*: A column bandwidth limiter sets the read and write bandwidth of
  each array column. Each weight is then passed to a row group in the column.
- *Output Path*: Outputs also pass through a column bandwidth limiter to set the
  read and write bandwidth of each array column.
  
Next, there are 16 row groups in the column. Each row group contains 8 rows of
SRAM cells. Outputs are reused between row groups (*i.e.,* the output of one row
group is propagated to the next for further accumulation), while inputs and
weights are not reused.

### Row Group Level

- *Input Path*: Each input is passed directly to a row in the row group.
- *Weight Path*: Each weight is passed directly to a row in the row group.
- *Output Path*: Outputs pass through a register. A register, in the form of a
  flip-flop for each column of the array, reduces the critical path length of
  the array and to allow for a faster pipelined operation.
  
Next, there are 8 rows in each row group. Outputs are reused between rows
(*i.e.,* are propagated from the full adders of one row to the next), while
inputs and weights are not reused.

### Row Level

- *Input Path*: The input is used for a MAC operation.
- *Weight Path*: A 1b weight is stored in the one SRAM cell in the CiM unit in
  the row. Additional digital logic multiplies the input & weight, computes 1b
  MAC operations, and accumulates outputs from adjacent rows and columns.
- *Output Path*: The output is used for a MAC operation.

Inside the CiM unit, one or more virtualized MAC units compute the MAC
operation. Note that the number of virtualized MAC units may change depending on
the number of bits of input and weight that are being used.

In [ ]:
from _scripts import *
from tqdm.auto import tqdm

display_important_variables("colonnade_jssc_2021")
get_spec("colonnade_jssc_2021").arch

### Area Breakdown

This test replicates the results presented in Fig. 17 of the paper.

We show the area breakdown fo the macro. The area is broken down into the
following components:

- Bitcell Array: The energy consumed by the SRAM cells and digital adders in
  the array.
- Drivers: The energy consumed by the array drivers. Note that we model a
  transposed version of the macro, so our drivers are modeled as wordline
  drivers while the Colonnade paper uses bitline drivers.
- Ctrl+Reg: The energy consumed by the control logic and registers.
  Registers include those inside the array that accomplish in-array
  pipelining.
- BL Decoder: Note that in Colonnade this is actually a WL decoder, but
  we're transposed in this model.

We can see that the bitcell array consumes the majority of the macro's area.
This is because the digital CiM macro requires a full adder and
configuration circuits to be connected to every SRAM bitcell, resulting in
high area and energy overhead for the SRAM array.

In [ ]:
total_ref_area = 227200e-12
ref_fracs = {
    "Bitcell Array": 0.839,
    "Drivers": 0.018,
    "Ctrl+Reg": 0.125,
    "BL Decoder": 0.018,
}
groups = {
    "Bitcell Array": ["CimUnit", "CimLogic"],
    "Drivers": ["RowDrivers", "DigitalLogicInputPorts"],
    "Ctrl+Reg": ["Register"],
    "BL Decoder": ["ColumnDrivers"],
}

spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=False)
spec.arch.variables["force_100mhz"] = True
for einsum in spec.workload.einsums:
    for ta in einsum.tensor_accesses:
        ta.bits_per_value = {"input": 1, "weight": 8, "output": 16}[ta.name]

area = {
    k: float(v)
    for k, v in spec.calculate_component_costs().arch.per_component_total_area.items()
}
modeled_um2 = {g: sum(area[c] for c in cs) * 1e12 for g, cs in groups.items()}
reference_um2 = {k: total_ref_area * f * 1e12 for k, f in ref_fracs.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bar_comparison(
    {"Reference": reference_um2, "Modeled": modeled_um2},
    "Component",
    "Area (um²)",
    "Area Breakdown (Fig. 17)",
    ax,
)
plt.tight_layout()
plt.show()

print(f"{'Group':<20} {'Modeled':>12} {'Reference':>12} {'Ratio':>8}")
for k in ref_fracs:
    m, r = modeled_um2[k], reference_um2[k]
    print(f"{k:<20} {m:10.0f} um² {r:10.0f} um² {m/r:8.2f}x")

### Energy Breakdown

This test replicates the results presented in Fig. 17 of the paper. Energy
is measured using 8b weights.

We show the energy breakdown fo the macro. The energy is broken down into
the following components:

- Bitcell Array: The energy consumed by the SRAM cells and digital CiM
  circuitry (1b multiply, adders, configuration) in the array.
- Drivers: The energy consumed by the array drivers. Note that we model a
  transposed version of the macro, so our drivers are modeled as wordline
  drivers while the Colonnade paper uses bitline drivers.
- Ctrl+Reg: The energy consumed by the control logic and registers.
  Registers include those inside the array that accomplish in-array
  pipelining.
- BL Decoder: Note that in Colonnade this is actually a WL decoder, but
  we're transposed in this model.

We can see that the bitcell array consumes the majority of the macro's energy. This is
because the digital CiM macro requires a full adder and configuration circuits to be
connected to every SRAM bitcell, resulting in high area and energy overhead for the SRAM
array.

In [ ]:
ref_total_pj = 0.1443
ref_fracs_e = {
    "Bitcell Array": 0.602,
    "Drivers": 0.288,
    "Ctrl+Reg": 0.075,
    "BL Decoder": 0.035,
}
groups = {
    "Bitcell Array": ["CimUnit", "CimLogic"],
    "Drivers": ["RowDrivers", "DigitalLogicInputPorts"],
    "Ctrl+Reg": ["Register"],
    "BL Decoder": ["ColumnDrivers"],
}

spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=True)
spec.arch.variables["force_100mhz"] = True
for einsum in spec.workload.einsums:
    for ta in einsum.tensor_accesses:
        ta.bits_per_value = {"input": 1, "weight": 8, "output": 16}[ta.name]

energy = (
    spec.map_workload_to_arch(print_progress=False)
    .per_compute()
    .energy(per_component=True)
)
modeled_pj = {
    g: sum(float(energy[c]) for c in cs if c in energy) * 1e12
    for g, cs in groups.items()
}
reference_pj = {k: ref_total_pj * f for k, f in ref_fracs_e.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bar_comparison(
    {"Reference": reference_pj, "Modeled": modeled_pj},
    "Component",
    "Energy (pJ/MAC)",
    "Energy Breakdown (Fig. 17) at 0.8V",
    ax,
)
plt.tight_layout()
plt.show()

print(f"{'Group':<20} {'Modeled':>12} {'Reference':>12} {'Ratio':>8}")
for k in ref_fracs_e:
    m, r = modeled_pj[k], reference_pj[k]
    print(f"{k:<20} {m:12.5f} pJ {r:12.5f} pJ {m/r:8.2f}x")

### Energy vs. Number of Weight Bits

This test replicates the results presented in Fig. 16b of the paper. The
macro supports between 1 and 16 weight bits. For three different voltages,
the energy per MAC of the macro is measured for varying numbers of weight
bits. We fix the number of input bits to 1. We also fix the clock frequency
at 100MHz, as is done in the paper (for measurement purposes; the macro can
support higher frequencies).

We see that the number of weight bits scales energy because of two factors.

- First, as the number of weight bits increases, the number of adders
  required to compute MAC results increases, leading to higher energy
  consumption. The number of full adders required to compute each MAC is
  equal to the number of weight bits plus 7. This is because the weights are
  stored in the array and the accumulation results of up to 128 MACs are
  propagated through adders. The array therefore requires one adder for
  each weight bit plus an additional 7 adders per weight to store the
  full precision of the accumulation results.
- Second, as the number of weight bits increases, the number of parallel
  MACs that the array can perform decreases. This will lead to a decrease in
  the number of parallel MACs, and thus an increase in energy per MAC. This
  effect leads to steps in the energy each time the number of parallel MACs
  decreases. For example, there is a step when going from 14b (21 columns
  per weight, 6 parallel MACs per row) to 15b weights (22 columns per
  weight, 5 parallel MACs per row).

We also see that the energy per MAC increases with voltage.

There are two significant differences between published and modeled results:

- In the published results, the energy per MAC does not increase
  monotonically with the number of weight bits. Energy per MAC may decrease
  from N to N+1 bit weights if the number of parallel MACs is the same for
  both N and N+1 bit weights (i.e., the array utilization is the same for
  both). We attribute this to a change in data-value-dependent switching
  activity of adders in the array, and this effect could be captured with a
  more complex data-value-dependent calculation.
- In the published results, the energy per MAC decreases rapidly as supply
  voltage decreases (~1.6x from 0.8V to 0.7V and ~4x from 0.8V to 0.6V).
  This is likely due to some technology-specific effects. We could model it
  by using a piecewise function for VOLTAGE_ENERGY_SCALE and passing it to
  all subcomponent models, but we instead chose to use simple V^2 scaling.

In [ ]:
expected_fj_per_mac_w = {
    #    0.6V          0.7V           0.8V
    1: [17.07820589, 42.83967039, 68.31697711],
    2: [20.27486225, 49.48109179, 77.83242912],
    3: [24.0698594, 59.9648546, 96.94840357],
    4: [26.49734159, 66.92475081, 105.2708593],
    5: [29.37052503, 73.6742141, 115.0949303],
    6: [33.23249487, 83.93585286, 131.125787],
    7: [30.81598573, 77.30009326, 121.5909692],
    8: [36.83598758, 91.76892401, 144.3500315],
    9: [34.39268536, 85.09591799, 133.8535776],
    10: [42.83967039, 106.725807, 166.7286341],
    11: [39.72457364, 99.64675226, 158.9079843],
    12: [51.56119566, 126.7024855, 204.8466723],
    13: [49.82184952, 121.5909692, 189.9512015],
    14: [47.16012347, 115.0949303, 181.041186],
    15: [63.7855784, 157.8210593, 255.1576938],
    16: [59.9648546, 151.4542256, 243.1891829],
}

voltages = [0.6, 0.7, 0.8]


def _run_colonnade_w_bits(weight_bits, voltage):
    spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=True)
    spec.variables.voltage = voltage
    spec.arch.variables["force_100mhz"] = True
    # parallel_macs = floor(128 / encoded_weight_bits) is the number of distinct
    # output values computed per cycle (each spans encoded_weight_bits columns).
    encoded_weight_bits = weight_bits + 7
    parallel_macs = 128 // encoded_weight_bits
    spec.workload.rank_sizes["N"] = parallel_macs
    for einsum in spec.workload.einsums:
        for ta in einsum.tensor_accesses:
            if ta.name == "input":
                ta.bits_per_value = 1
            elif ta.name == "weight":
                ta.bits_per_value = weight_bits
            elif ta.name == "output":
                ta.bits_per_value = 16
    r = spec.map_workload_to_arch(print_progress=False)
    per_comp = {
        k: float(v) * 1e15
        for k, v in r.per_compute().energy(per_component=True).items()
    }
    return weight_bits, voltage, r.per_compute().energy() * 1e15, per_comp


out_w = parallel(
    [
        delayed(_run_colonnade_w_bits)(b, v)
        for b in expected_fj_per_mac_w
        for v in voltages
    ],
    pbar="Weight bits sweep",
)

mod_by_v_w = {v: {} for v in voltages}
mod_by_v_w_per_comp = {v: {} for v in voltages}
for b, v, e, per_comp in out_w:
    mod_by_v_w[v][b] = e
    mod_by_v_w_per_comp[v][b] = per_comp

fig, axs = plt.subplots(1, len(voltages), figsize=(5 * len(voltages), 5))
xs = sorted(expected_fj_per_mac_w)
for i, v in enumerate(voltages):
    ref = [expected_fj_per_mac_w[b][i] for b in xs]
    mod = [mod_by_v_w[v][b] for b in xs]
    axs[i].plot(xs, ref, "o--", label="Reference")
    axs[i].plot(xs, mod, "s-", label="Modeled")
    axs[i].set_xlabel("Number of Weight Bits")
    axs[i].set_ylabel("Energy per MAC (fJ)")
    axs[i].set_title(f"Energy Scaling with Weight Bits at {v}V")
    axs[i].legend()
    axs[i].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, len(voltages), figsize=(5 * len(voltages), 5))
xs = sorted(expected_fj_per_mac_w)
for i, v in enumerate(voltages):
    stacked = {
        f"{b}b": {k: val for k, val in mod_by_v_w_per_comp[v][b].items() if val > 0}
        for b in xs
    }
    bar_stacked(
        stacked,
        "Number of Weight Bits",
        "Energy per MAC (fJ)",
        f"Energy Scaling with Weight Bits at {v}V (per-component)",
        axs[i],
    )
plt.tight_layout()
plt.show()

# Energy vs. Number of Input and Weight Bits

This test replicates the results presented in Fig. 16a of the paper. Like
with the energy versus weight bits test, we'll test three voltages and
measure energy 1-16b weights. This time, we vary the number of input bits as
well, using the same number of input bits as weight bits.

We can see that, in addition to the energy increasing with the number of
weight bits, the energy also increases linearly with the number of input
bits because an N-bit input is processed in N cycles and N activations of
the array.  We also fix the clock frequency at 100MHz, as is done in the
paper (for measurement purposes; the macro can support higher frequencies).

Modeled and reference results vary for the same two reasons as the energy
versus weight bits test, and could be made closer by using a more complex
data-value-dependent calculation for the energy of the adders in the array,
and by using a more complex voltage-energy scaling function.

In [ ]:
expected_fj_per_mac_iw = {
    #    0.6V          0.7V           0.8V
    1: [17.1320369, 42.5577975, 68.10051312],
    2: [39.750389, 101.7850194, 161.6449048],
    3: [72.35941721, 183.8842178, 292.0267348],
    4: [104.1269428, 260.6311646, 423.4320763],
    5: [143.1768052, 369.409647, 591.1251988],
    6: [198.3696842, 507.9462783, 788.5272112],
    7: [215.6250592, 543.8204606, 844.217685],
    8: [292.0267348, 725.4253956, 1169.653111],
    9: [310.2896494, 788.5272112, 1252.260958],
    10: [420.2334993, 1067.923309, 1695.970219],
    11: [439.7938751, 1109.188833, 1748.197762],
    12: [613.9668116, 1536.767343, 2422.106272],
    13: [632.8739703, 1584.09222, 2554.140514],
    14: [642.5447846, 1645.302977, 2632.795482],
    15: [960.366235, 2459.11797, 3905.328036],
    16: [989.9407926, 2515.698651, 3995.183881],
}


def _run_colonnade_iw_bits(bits, voltage):
    spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=True)
    spec.variables.voltage = voltage
    spec.arch.variables["force_100mhz"] = True
    # See _run_colonnade_w_bits in the previous cell for details on N=parallel_macs.
    encoded_weight_bits = bits + 7
    parallel_macs = 128 // encoded_weight_bits
    spec.workload.rank_sizes["N"] = parallel_macs
    for einsum in spec.workload.einsums:
        for ta in einsum.tensor_accesses:
            if ta.name == "input":
                ta.bits_per_value = bits
            elif ta.name == "weight":
                ta.bits_per_value = bits
            elif ta.name == "output":
                ta.bits_per_value = 16
    r = spec.map_workload_to_arch(print_progress=False)
    return bits, voltage, r.per_compute().energy() * 1e15


out_iw = parallel(
    [
        delayed(_run_colonnade_iw_bits)(b, v)
        for b in expected_fj_per_mac_iw
        for v in voltages
    ],
    pbar="Input+weight bits sweep",
)

mod_by_v_iw = {v: {} for v in voltages}
for b, v, e in out_iw:
    mod_by_v_iw[v][b] = e

fig, axs = plt.subplots(1, len(voltages), figsize=(5 * len(voltages), 5))
xs = sorted(expected_fj_per_mac_iw)
for i, v in enumerate(voltages):
    ref = [expected_fj_per_mac_iw[b][i] for b in xs]
    mod = [mod_by_v_iw[v][b] for b in xs]
    axs[i].plot(xs, ref, "o--", label="Reference")
    axs[i].plot(xs, mod, "s-", label="Modeled")
    axs[i].set_xlabel("Input and Weight Bits")
    axs[i].set_ylabel("Energy (fJ/MAC)")
    axs[i].set_title(f"Energy Scaling with Input/Weight Bits at {v}V")
    axs[i].legend()
    axs[i].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Throughput vs. Number of Registers per Column

This test replicates the results presented in Fig. 15a of the paper. The
number of registers in the macro is varied from 1 to 16. A register in the
macro is a flip flop, one for each column in the array, that breaks up the
critical path length into a smaller number of rows to allow for higher clock
frequencies.

In addition to varying the number of registers, the number of input and
weight bits are varied from 1 to 16.

Throughput increases with the number of registers as the critical path
latency reduces. Effects begin to saturate with many registers as the
critical path begins to become dominated by the full adders communicating
between columns rather than between rows (i.e., carry signals propagating
between adders in the same row, rather than sum signals propagating between
adders in the same column).

We also find that throughput decreases approximately quadratically as we
increase the number of input and weight bits because the number of clock
cycles increases linearly with the number of input bits and the number of
computations computed by the array decreases approximately linearly with the
number of weight bits.

In [ ]:
expected_throughput = {
    # 1 reg   2 regs   4 regs   8 regs   16 regs
    1: [75.07, 138.51, 243.93, 391.42, 563.50],
    2: [32.50, 60.42, 104.77, 168.13, 238.32],
    3: [18.31, 34.04, 59.03, 92.55, 127.18],
    4: [12.52, 23.10, 39.45, 60.42, 83.03],
    5: [9.11, 16.43, 28.05, 42.96, 58.12],
    6: [6.73, 12.33, 20.41, 31.26, 41.65],
    7: [5.81, 10.24, 17.21, 25.75, 34.84],
    8: [4.43, 7.92, 13.32, 19.48, 25.75],
    9: [3.91, 6.95, 11.41, 16.81, 22.23],
    10: [3.08, 5.38, 8.77, 12.72, 16.68],
    11: [2.76, 4.79, 7.99, 11.50, 14.51],
    12: [2.14, 3.74, 6.09, 8.56, 11.06],
    13: [1.99, 3.48, 5.46, 7.62, 9.77],
    14: [1.82, 3.13, 4.94, 6.95, 8.90],
    15: [1.43, 2.42, 3.79, 5.25, 6.63],
    16: [1.34, 2.26, 3.51, 4.79, 5.99],
}

regs_list = [1, 2, 4, 8, 16]


# AccelForge requires spatial fanouts as integer literals at YAML load time,
# so each sweep iteration reloads the spec and patches fanouts via find_spatial.
def _run_colonnade_throughput(input_bits, n_regs_per_col):
    n_rows_per_reg = 128 // n_regs_per_col
    spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=True)
    spec.arch.variables["force_100mhz"] = False
    spec.arch.variables["n_regs_per_col"] = n_regs_per_col
    spec.arch.variables["n_rows_per_reg"] = n_rows_per_reg
    spec.arch.find_spatial("group_ARRAY_ROWS").fanout = n_regs_per_col
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = n_rows_per_reg
    # See _run_colonnade_w_bits for details on N=parallel_macs.
    encoded_weight_bits = input_bits + 7
    parallel_macs = 128 // encoded_weight_bits
    spec.workload.rank_sizes["N"] = parallel_macs
    for einsum in spec.workload.einsums:
        for ta in einsum.tensor_accesses:
            if ta.name in ("input", "weight"):
                ta.bits_per_value = input_bits
            elif ta.name == "output":
                ta.bits_per_value = 16
    r = spec.map_workload_to_arch(print_progress=False)
    return input_bits, n_regs_per_col, 2 / r.per_compute().latency() / 1e12


out_t = parallel(
    [
        delayed(_run_colonnade_throughput)(b, n)
        for b in expected_throughput
        for n in regs_list
    ],
    pbar="Throughput sweep",
)

mod_by_n_t = {n: {} for n in regs_list}
for b, n, t in out_t:
    mod_by_n_t[n][b] = t

fig, axs = plt.subplots(1, len(regs_list), figsize=(5 * len(regs_list), 5))
xs = sorted(expected_throughput)
for i, n in enumerate(regs_list):
    ref_tops = {b: expected_throughput[b][i] / 1000 for b in xs}
    mod_tops = {b: mod_by_n_t[n][b] for b in xs}
    bar_comparison(
        {"Reference": ref_tops, "Modeled": mod_tops},
        "Input and Weight Bits",
        "Throughput (TOPS)",
        f"# Input/Weight Bits vs. Throughput\n with {n} register(s) per column",
        axs[i],
    )
plt.tight_layout()
plt.show()

### Latency vs. Number of Registers per Column

This test replicates the results presented in Fig. 16b of the paper. Like in
the previous test, the number of registers in the macro is varied from 1 to
16. The latency of the macro is measured for varying numbers of registers.

Note that in other tests, the macro is bit pipelined, meaning that when one
pipeline stage (a stage being a row group in the array) finishes processing
a bit, it is immediately filled with the next bit even if all pipeline stages
are not finished processing the current bit. In this test, the macro is NOT
bit pipelined, meaning that all pipeline stages need to be flushed before
processing the next bit. This significantly increases the latency of the
macro.

We find first that latency increases approximately linearly with the number
of input bits because each additional input bit requires an additional clock
cycle to process. We find that, at first, latency decreases with additional
registers because the critical path length of the array is reduced. However,
at a certain point (2-4 registers depending on the number of input bits),
latency begins to increase because flushing the pipeline requires additional
clock cycles.

In [ ]:
expected_latency = {
    # 1 reg   2 regs   4 regs   8 regs   16 regs
    1: [0.12, 0.09, 0.09, 0.10, 0.13],
    2: [0.22, 0.18, 0.17, 0.20, 0.26],
    3: [0.34, 0.27, 0.26, 0.30, 0.41],
    4: [0.46, 0.37, 0.37, 0.42, 0.58],
    5: [0.57, 0.47, 0.46, 0.54, 0.75],
    6: [0.68, 0.57, 0.56, 0.67, 0.95],
    7: [0.80, 0.67, 0.67, 0.82, 1.15],
    8: [0.94, 0.78, 0.79, 0.96, 1.37],
    9: [1.05, 0.88, 0.91, 1.11, 1.59],
    10: [1.17, 1.00, 1.03, 1.26, 1.85],
    11: [1.30, 1.11, 1.15, 1.44, 2.09],
    12: [1.44, 1.22, 1.27, 1.61, 2.37],
    13: [1.55, 1.34, 1.41, 1.78, 2.66],
    14: [1.69, 1.46, 1.54, 1.96, 2.95],
    15: [1.82, 1.58, 1.69, 2.16, 3.27],
    16: [1.95, 1.71, 1.82, 2.36, 3.60],
}


def _run_colonnade_latency(input_bits, n_regs_per_col):
    n_rows_per_reg = 128 // n_regs_per_col
    spec = get_spec("colonnade_jssc_2021", add_dummy_main_memory=True)
    spec.arch.variables["force_100mhz"] = False
    # bit_pipelined=False is what distinguishes this test from throughput_scaling.
    spec.arch.variables["bit_pipelined"] = False
    spec.arch.variables["n_regs_per_col"] = n_regs_per_col
    spec.arch.variables["n_rows_per_reg"] = n_rows_per_reg
    spec.arch.find_spatial("group_ARRAY_ROWS").fanout = n_regs_per_col
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = n_rows_per_reg
    encoded_weight_bits = input_bits + 7
    parallel_macs = 128 // encoded_weight_bits
    spec.workload.rank_sizes["N"] = parallel_macs
    for einsum in spec.workload.einsums:
        for ta in einsum.tensor_accesses:
            if ta.name in ("input", "weight"):
                ta.bits_per_value = input_bits
            elif ta.name == "output":
                ta.bits_per_value = 16
    r = spec.map_workload_to_arch(print_progress=False)
    return input_bits, n_regs_per_col, r.latency() * 1e6


out_l = parallel(
    [
        delayed(_run_colonnade_latency)(b, n)
        for b in expected_latency
        for n in regs_list
    ],
    pbar="Latency sweep",
)

mod_by_n_l = {n: {} for n in regs_list}
for b, n, t in out_l:
    mod_by_n_l[n][b] = t

fig, axs = plt.subplots(1, len(regs_list), figsize=(5 * len(regs_list), 5))
xs = sorted(expected_latency)
for i, n in enumerate(regs_list):
    ref_lat = {b: expected_latency[b][i] for b in xs}
    mod_lat = {b: mod_by_n_l[n][b] for b in xs}
    bar_comparison(
        {"Reference": ref_lat, "Modeled": mod_lat},
        "Input and Weight Bits",
        "Latency (us/Full Array of MACs)",
        f"# Input/Weight Bits vs. Latency\n with {n} register(s) per column\nNon-Bit-Pipelined",
        axs[i],
    )
plt.tight_layout()
plt.show()